# 07. 대본 LLM 의역 + 검증 — **Colab GPU (A100·G4)**

템플릿 대본에서 **판정 발화(call)와 해설위원 코멘트(comment, distract)만** 의역한다. **Hard 트랙만** 의역하고 Easy는 템플릿 그대로.
이닝·투수·타자 소개, 카운트 재언급, 공수 교대는 정답 단서라 그대로 둔다. 규칙과 프롬프트는 `kbolisten/paraphrase.py`.

**`MODE`** (셀 2): `"pilot"` = 설계 검증용 20클립, `"main"` = 본 데이터셋 1,300클립 (train 1000 / dev 100 / test 200).
클립·목소리·트랙 노브는 `kbolisten/build.py` — 06·08과 같은 MODE면 같은 클립이 나온다.

| 단계 | 모델 | 내용 |
|---|---|---|
| 1. 생성 | Qwen3-32B | 클립 하나를 한 번에 의역 → 형식 검사 → 탈락한 줄만 최대 2번 다시 생성 |
| 2. 검증 | Gemma 3 27B | 원문·정답을 모르는 채 문장만 보고 판정 라벨 분류 → 원래 판정과 같아야 통과 |

생성과 검증을 다른 계열 모델로 하는 이유: 같은 모델이면 같은 착각을 같이 해서 오류가 통과한다.
끝까지 실패한 줄은 템플릿 원문을 쓴다 (클립은 버리지 않음).

**실행 전 준비**
- 새 런타임 권장 (vLLM이 PyTorch 버전을 바꿔서 qwen-tts와 꼬일 수 있음)
- `kbolisten/`은 Drive for desktop으로 동기화한 폴더(`Othercomputers/*/kbolisten`)에서 자동으로 불러온다
- Gemma는 승인이 필요한 모델: huggingface.co/google/gemma-3-27b-it 에서 라이선스 동의 →
  Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN` 등록
- 두 모델이 GPU에 같이 안 올라가서 **1단계 뒤 세션 다시 시작** → 셀 2 → 2단계 순서로 실행

**결과물 (Drive `kbo-listen/paraphrase_{pilot|main}/`)**: `generated.jsonl`(1단계), `scripts.jsonl`(최종 대본), `report.json`

In [1]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/

In [3]:
# 공통 준비: 1단계·2단계 모두 커널 시작 후 이 셀부터 실행 (샘플링·대본은 시드 고정이라 매번 같음)
MODE = "main"  # "pilot"(설계 검증 20클립) 또는 "main"(본 데이터셋 1,300클립, Hard 650클립 의역)

import glob
import json
import os
import random
import sys
import zlib
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

from google.colab import drive

# kbolisten은 Drive for desktop으로 동기화한 로컬 폴더에서 불러온다 (매번 업로드할 필요 없음)
drive.mount("/content/drive")
_found = glob.glob("/content/drive/Othercomputers/*/kbolisten")
assert _found, "Drive '컴퓨터' 영역에서 kbolisten을 못 찾음 → Drive for desktop 동기화 확인"
sys.path.insert(0, os.path.dirname(_found[0]))

from kbolisten import build as B
from kbolisten import paraphrase as P

GEN_MODEL = "Qwen/Qwen3-32B"            # 생성: 라이선스(Apache 2.0) 문제 없는 모델 중 한국어가 가장 자연스러움
JUDGE_MODEL = "google/gemma-3-27b-it"   # 검증: 생성과 다른 계열


@contextmanager
def real_stdout():
    """vLLM은 엔진을 띄울 때 sys.stdout.fileno()를 부르는데, 노트북 커널의 출력 스트림은 fileno가 없어서 실패한다.
    엔진을 만드는 동안만 원래 터미널 출력으로 바꿔 둔다 (이 동안의 로그는 셀이 아니라 런타임 로그로 감)"""
    saved = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        yield
    finally:
        sys.stdout, sys.stderr = saved


DRIVE = Path("/content/drive/MyDrive/kbo-listen")
OUT = DRIVE / B.DIRS[MODE]["paraphrase"]
OUT.mkdir(parents=True, exist_ok=True)

VOICES = json.loads((DRIVE / "voices" / "voices.json").read_text(encoding="utf-8"))
with (DRIVE / "labels_all.jsonl").open(encoding="utf-8") as f:
    clips = [json.loads(line) for line in f]

# 샘플링·트랙 노브·목소리 배정·템플릿 대본은 kbolisten/build.py (06·08과 같은 결과)
picked = B.prepare(clips, MODE, VOICES)
for c in picked:
    # 의역할 발화 → 기대 라벨 (의역하지 않는 트랙은 비워 둠 → 템플릿 그대로 저장)
    c["todo"] = P.targets(c["utts"], c["events"]) if c["knobs"]["paraphrase"] else {}
    c["repeat_ok"] = P.repeat_ok(c["utts"], c["events"])     # '또 파울'처럼 반복 표현을 써도 사실인 발화

print(MODE, "→", OUT.name)
print(f"{len(picked)}클립 {Counter((c['split'], c['track']) for c in picked)}")
print(f"발화 {sum(len(c['utts']) for c in picked)}개 중 의역 대상 {sum(len(c['todo']) for c in picked)}개 (Hard만)")
print(Counter(l for c in picked for l in c["todo"].values()))

Mounted at /content/drive
main → paraphrase_main
1300클립 Counter({('train', 'easy'): 500, ('train', 'hard'): 500, ('test', 'easy'): 100, ('test', 'hard'): 100, ('dev', 'easy'): 50, ('dev', 'hard'): 50})
발화 44529개 중 의역 대상 13664개 (Hard만)
Counter({'none': 4762, 'ball': 3010, 'strike': 1833, 'foul': 1384, 'out': 1101, 'reach': 736, 'strikeout': 462, 'walk': 337, 'double_play': 39})


## 1단계: 생성 (Qwen3-32B)

In [ ]:
from vllm import LLM, SamplingParams

ROUNDS = 3          # 첫 생성 + 탈락한 줄 다시 생성 2번
TEMPERATURE = 0.7   # 0.9에서는 '드러눕니다' 같은 이상한 단어 선택이 잦았음

# 32B bf16은 가중치만 약 65GB → 남는 메모리로 KV 캐시. 파일럿·본 데이터 모두 프롬프트가 짧아 충분
# 이미 모델을 띄웠다면(같은 커널에서 다시 실행) 새로 만들지 않는다
if "llm" not in globals():
    with real_stdout():
        llm = LLM(GEN_MODEL, dtype="bfloat16", max_model_len=8192, gpu_memory_utilization=0.93, seed=0)
print("모델 준비 완료:", GEN_MODEL)


def form_check(c, i, new):
    u = c["utts"][i]
    return P.check_form(u["text"], new, u["kind"], repeat_ok=i in c["repeat_ok"], label=c["todo"][i])


accepted = {c["clip_id"]: {} for c in picked}   # clip_id → {발화 번호: 의역}
rejects = Counter()
examples = defaultdict(list)                     # 탈락 이유 → 예시 몇 개
for r in range(ROUNDS):
    jobs = [(c, [i for i in c["todo"] if i not in accepted[c["clip_id"]]]) for c in picked]
    jobs = [(c, todo) for c, todo in jobs if todo]
    if not jobs:
        break
    # 앞 라운드에서 통과한 의역은 문맥으로 보여줘서 말투가 이어지게 한다
    msgs = [P.gen_messages(P.apply(c["utts"], accepted[c["clip_id"]]), set(todo), c["voices"]) for c, todo in jobs]
    params = [SamplingParams(temperature=TEMPERATURE, top_p=0.95, max_tokens=3000,
                             seed=zlib.crc32(f"{c['clip_id']}/{r}".encode())) for c, _ in jobs]
    outs = llm.chat(msgs, params, chat_template_kwargs={"enable_thinking": False})

    n_try = n_ok = 0
    for (c, todo), out in zip(jobs, outs):
        got = P.parse_gen(out.outputs[0].text)
        for i in todo:
            n_try += 1
            new = got.get(i)
            reason = "응답에 없음" if new is None else form_check(c, i, new)
            if reason is None:
                accepted[c["clip_id"]][i] = new
                n_ok += 1
            else:
                rejects[reason] += 1
                if len(examples[reason]) < 3:
                    examples[reason].append(f"{c['utts'][i]['text']}  →  {new}")
    print(f"라운드 {r + 1}: {n_try}줄 중 {n_ok}줄 통과")

total = sum(len(c["todo"]) for c in picked)
done = sum(len(a) for a in accepted.values())
print(f"\n형식 검사 통과 {done}/{total} ({done / total:.0%})")
for reason, n in rejects.most_common():
    print(f"  {n:>4}  {reason}")
    for ex in examples[reason]:
        print(f"         {ex}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Processed prompts: 100%|██████████| 594/594 [03:14<00:00,  3.05it/s, est. speed input: 6479.75 toks/s, output: 283.77 toks/s]

라운드 2: 2076줄 중 1440줄 통과


Rendering conversations:   0%|          | 0/356 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 356/356 [01:49<00:00,  3.24it/s, est. speed input: 7048.78 toks/s, output: 216.25 toks/s]

라운드 3: 636줄 중 353줄 통과

형식 검사 통과 13381/13664 (98%)
   595  방해 코멘트에 판정 단어 없음
         앞선 볼도 비슷한 코스였죠.  →  이미 몇 번이나 비슷한 코스로 던졌습니다.
         아까 그 공은 볼 판정이 조금 아쉬웠어요.  →  아까 그 공은 정말 슬라이드가 좋았던 것 같아요.
         앞선 볼도 비슷한 코스였죠.  →  이전 공도 거의 똑같은 코스였습니다.
   536  근거 없는 반복 표현
         파울이고요.  →  다시 뒤쪽으로 빗나갑니다, 파울.
         파울이고요.  →  이번에도 뒤쪽으로 빗나갑니다, 파울.
         파울.  →  이번에도 뒤로 보내 버립니다.
   481  응답에 없음
         파울로 계속 걷어내는 게 타자 입장에선 최선이에요.  →  None
         파울로 계속 걷어내는 게 타자 입장에선 최선이에요.  →  None
         초구 스트라이크가 정말 중요하거든요.  →  None
   362  없던 숫자
         직구, 잡아냅니다, 아웃.  →  직구로 정면을 찍습니다, 3루수 아웃.
         슬라이더, 바깥쪽 볼이고요.  →  132킬로 슬라이더, 바깥쪽 볼입니다.
         뜬공, 잡았습니다.  →  타구는 공중으로 떴고, 2루수 손에 잡힙니다.
   335  없던 구종
         자, 일단 빠집니다.  →  빠져나가는 직구, 루킹 아웃입니다.
         높은 스트라이크.  →  스트라이크존 정중앙, 높은 직구가 정확하게 찍혔습니다.
         자, 슬라이더, 루킹 스트라이크.  →  체인지업, 루킹 스트라이크.
   195  너무 김
         출루에 성공했습니다.  →  파울을 두 번 잡고 들어온 타석, 결국 출루를 성공했죠.
         자, 초구 직구, 볼.  →  첫 공, 직구로 뻗어 나갑니다. 바깥쪽으로 살짝 벗어납니다.
         쳤습니다! 안타! 

In [ ]:
# 1단계 결과 저장 → 커널 재시작 후 2단계에서 읽음
with (OUT / "generated.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        f.write(json.dumps({"clip_id": c["clip_id"], "accepted": accepted[c["clip_id"]]}, ensure_ascii=False) + "\n")

# 클립 하나 미리보기: 원문 → 의역
c = next(c for c in picked if c["track"] == "hard")
acc = accepted[c["clip_id"]]
for i, u in enumerate(c["utts"]):
    who = "해설" if u["speaker"] == "analyst" else "캐스터"
    if i in acc:
        print(f"{who:<3} {u['text']}\n      → {acc[i]}")
    else:
        print(f"{who:<3} {u['text']}" + ("" if i not in c["todo"] else "   (실패 → 원문)"))

캐스터 3회말, LG 공격입니다. 스코어 키움 2 대 LG 2.
캐스터 마운드에는 후라도.
캐스터 노아웃, 타석에는 오스틴.
캐스터 초구 131킬로 커브, 몸쪽 스트라이크입니다.
      → 131킬로 커브, 몸쪽으로 들어옵니다. 스트라이크.
캐스터 노볼 원 스트라이크입니다.
해설  그렇죠.
캐스터 바깥쪽 빠집니다.
      → 바깥쪽으로 빠지면서 타자가 몸을 피웠습니다.
해설  원아웃 이후의 실점이 제일 뼈아프거든요.
      → 하나의 아웃만 있다가 실점하면 팀 분위기에 큰 영향을 줍니다.
캐스터 체인지업, 낮은 볼입니다.
      → 체인지업, 낮게 떨어지며 볼 판정.
해설  유인구에 잘 참았습니다.
      → 유인구를 노리며 잘 참았어요. 집중도가 높아 보입니다.
캐스터 커트해냅니다.
      → 타격 포인트를 벗어나 파울로 처리합니다.
캐스터 중견수 앞 안타.
      → 중견수 쪽 앞쪽으로 떨어집니다. 안타!
해설  공략을 잘했습니다.
      → 투수의 구종 변화를 읽고 타격했죠. 잘 공략했습니다.
캐스터 노아웃, 타석에는 오지환.
캐스터 초구 직구, 헛스윙이고요.
      → 직구가 들어오자마자 헛스윙.
캐스터 볼카운트 노볼 원 스트라이크.
해설  파울로 계속 걷어내는 게 타자 입장에선 최선이에요.
      → 타자가 계속 파울로 벗어나는 게 좋습니다. 카운트를 잘 운영 중이죠.
캐스터 커터, 높게 빠졌고요.   (실패 → 원문)
캐스터 직구, 볼이고요.
      → 직구가 바깥쪽으로 흘러나갑니다. 볼.
해설  스트라이크 존을 넓게 쓰고 있어요.
      → 스트라이크 존 활용이 넓어 투수의 컨트롤이 안정적입니다.
캐스터 138킬로 커터, 볼.
      → 138킬로 커터, 바깥쪽으로 흘러나갑니다. 볼.
해설  그렇죠.
캐스터 그대로 들어오면서 스트라이크.
      → 중심을 지키며 들어옵니다. 스트라이크.
해설  초구 스트라이크가 정말 중요하거든요.
      → 초구로 스트라이크를 잡는 게 투수의 분위기를 바꿉니다.

## 2단계: 검증 (Gemma 3 27B)
**커널 재시작 → 셀 2(공통 준비) → 아래 셀** 순서로 실행. (1단계 모델이 GPU 메모리를 잡고 있음)

In [ ]:
from getpass import getpass

from vllm import LLM, SamplingParams

# Gemma는 라이선스 동의가 필요한 모델 → HF 토큰 필요
# Colab 웹에서는 보안 비밀(HF_TOKEN)을 읽고, VS Code처럼 보안 비밀을 못 읽는 환경에서는 직접 입력받는다
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        os.environ["HF_TOKEN"] = getpass("Hugging Face 토큰 (Read 권한): ").strip()

with real_stdout():
    llm = LLM(JUDGE_MODEL, dtype="bfloat16", max_model_len=2048, gpu_memory_utilization=0.9, seed=0)
print("모델 준비 완료:", JUDGE_MODEL)

with (OUT / "generated.jsonl").open(encoding="utf-8") as f:
    accepted = {}
    for line in f:
        row = json.loads(line)
        accepted[row["clip_id"]] = {int(i): t for i, t in row["accepted"].items()}

# 1단계 이후에 형식 검사 규칙이 늘었을 수 있으니 지금 규칙으로 한 번 더 거른다 (떨어진 줄은 템플릿 원문)
recheck = Counter()
for c in picked:
    acc = accepted[c["clip_id"]]
    for i in list(acc):
        u = c["utts"][i]
        reason = P.check_form(u["text"], acc[i], u["kind"], repeat_ok=i in c["repeat_ok"], label=c["todo"][i])
        if reason:
            recheck[reason] += 1
            print(f"  재검사 탈락 [{reason}] {u['text']}  →  {acc.pop(i)}")
print("재검사 탈락:", dict(recheck) or "없음")

# 의역문과 함께 원문도 분류한다 → 원문을 못 맞히는 라벨은 검증 모델 자체를 믿기 어렵다는 뜻
items = []  # (클립, 발화 번호, 원문/의역, 기대 라벨)
for c in picked:
    for i, label in c["todo"].items():
        items.append((c, i, "orig", label))
        if i in accepted[c["clip_id"]]:
            items.append((c, i, "para", label))


def utt_of(c, i, which):
    u = c["utts"][i]
    return u if which == "orig" else {**u, "text": accepted[c["clip_id"]][i]}


msgs = [P.judge_messages(utt_of(c, i, which)) for c, i, which, _ in items]
outs = llm.chat(msgs, SamplingParams(temperature=0, max_tokens=5))
preds = [P.parse_judge(o.outputs[0].text) for o in outs]

Hugging Face 토큰 (Read 권한): ··········


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

모델 준비 완료: google/gemma-3-27b-it
재검사 탈락: 없음


Rendering conversations:   0%|          | 0/27045 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 27045/27045 [01:11<00:00, 377.31it/s, est. speed input: 88053.41 toks/s, output: 1117.50 toks/s]


In [ ]:
# 검증 결과: 라벨별로 (원문 정확도 = 검증 모델 신뢰도) / (의역 통과율)
# 통과 기준은 P.judge_ok: 판정 발화는 원래 판정과 같아야 하고, 해설은 '정보 없음'이거나 원문과 같은 분류면 통과
orig_pred = {(c["clip_id"], i): pred for (c, i, which, _), pred in zip(items, preds) if which == "orig"}

stat = defaultdict(Counter)
final = {c["clip_id"]: {} for c in picked}
judge_fail = defaultdict(list)
for (c, i, which, label), pred in zip(items, preds):
    stat[label][f"{which}_n"] += 1
    if which == "orig":
        stat[label]["orig_ok"] += pred == label
        continue
    ok = P.judge_ok(label, pred, orig_pred[(c["clip_id"], i)])
    stat[label]["para_ok"] += ok
    if ok:
        final[c["clip_id"]][i] = accepted[c["clip_id"]][i]
    elif len(judge_fail[label]) < 4:
        judge_fail[label].append(f"{c['utts'][i]['text']}  →  {accepted[c['clip_id']][i]}   (분류: {pred})")

print(f"{'라벨':<12} {'원문 정확도':>10} {'의역 통과':>10}")
for label, s in sorted(stat.items()):
    print(f"{label:<12} {s['orig_ok']:>4}/{s['orig_n']:<5} {s['para_ok']:>4}/{s['para_n']:<5}")

total = sum(len(c["todo"]) for c in picked)
done = sum(len(v) for v in final.values())
print(f"\n최종 의역 반영 {done}/{total} ({done / total:.0%}), 나머지는 템플릿 원문")
print("\n검증에서 떨어진 예:")
for label, exs in judge_fail.items():
    print(f"[{label}]")
    for ex in exs:
        print("   ", ex)

라벨               원문 정확도      의역 통과
ball         3001/3010  2773/2982 
double_play    39/39      33/35   
foul         1255/1384  1137/1344 
none         3466/4762  4221/4608 
out          1101/1101  1027/1081 
reach         736/736    716/729  
strike       1833/1833  1599/1817 
strikeout     462/462    435/460  
walk          337/337    313/325  

최종 의역 반영 12254/13664 (90%), 나머지는 템플릿 원문

검증에서 떨어진 예:
[none]
    비거리가 충분했어요.  →  제법 길게 뻗어나가는 타구였죠.   (분류: reach)
    타자가 예상하지 못한 공이었네요.  →  타자 생각과는 다르게 흐르는 공이었어요.   (분류: ball)
    삼진 욕심보다는 맞혀 잡는 게 좋습니다.  →  볼넷을 얻는 게 때론 더 현명한 선택이죠.   (분류: walk)
    볼 배합이 좋네요.  →  볼을 골고루 써서 타자를 흔들고 있어요.   (분류: ball)
[out]
    번트, 희생번트 성공합니다.  →  번트 치고는 깔끔하게 들어갑니다.   (분류: reach)
    139킬로 커터, 뜬공, 잡았습니다.  →  139킬로 커터, 공이 뜨고 손에 잡힙니다.   (분류: ball)
    잡아냅니다, 아웃.  →  잡아냅니다, 타자 퇴장.   (분류: strikeout)
    자, 128킬로 체인지업, 평범한 타구, 아웃.  →  128킬로 체인지업, 타구는 정면이었고요.   (분류: ball)
[foul]
    파울.  →  계속 뒤로 걷어냅니다.   (분류: ball)
    자, 117킬로 커브, 빗맞으면서 파울.  →  117킬로 커브, 빗나가면서 뒤로 날아갑

In [ ]:
# 저장: 최종 대본(원문은 orig에 보존) + 보고서
import importlib.metadata as md

with (OUT / "scripts.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        row = {k: c[k] for k in ("clip_id", "split", "track", "tags", "voices", "knobs")}
        row["utts"] = P.apply(c["utts"], final[c["clip_id"]])
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

report = {
    "gen_model": GEN_MODEL, "judge_model": JUDGE_MODEL, "vllm": md.version("vllm"),
    "targets": total, "applied": done,
    "by_label": {label: dict(s) for label, s in stat.items()},
}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=1))

{
 "gen_model": "Qwen/Qwen3-32B",
 "judge_model": "google/gemma-3-27b-it",
 "vllm": "0.31.0",
 "targets": 13664,
 "applied": 12254,
 "by_label": {
  "strike": {
   "orig_n": 1833,
   "orig_ok": 1833,
   "para_n": 1817,
   "para_ok": 1599
  },
  "ball": {
   "orig_n": 3010,
   "orig_ok": 3001,
   "para_n": 2982,
   "para_ok": 2773
  },
  "none": {
   "orig_n": 4762,
   "orig_ok": 3466,
   "para_n": 4608,
   "para_ok": 4221
  },
  "foul": {
   "orig_n": 1384,
   "orig_ok": 1255,
   "para_n": 1344,
   "para_ok": 1137
  },
  "reach": {
   "orig_n": 736,
   "orig_ok": 736,
   "para_n": 729,
   "para_ok": 716
  },
  "strikeout": {
   "orig_n": 462,
   "orig_ok": 462,
   "para_n": 460,
   "para_ok": 435
  },
  "out": {
   "orig_n": 1101,
   "orig_ok": 1101,
   "para_n": 1081,
   "para_ok": 1027
  },
  "walk": {
   "orig_n": 337,
   "orig_ok": 337,
   "para_n": 325,
   "para_ok": 313
  },
  "double_play": {
   "orig_n": 39,
   "orig_ok": 39,
   "para_n": 35,
   "para_ok": 33
  }
 }
}


In [ ]:
# 사람 점검용: 반영된 의역을 라벨별로 무작위 몇 개씩
rng = random.Random(0)
by_label = defaultdict(list)
for c in picked:
    for i, t in final[c["clip_id"]].items():
        by_label[c["todo"][i]].append(f"{c['utts'][i]['text']}  →  {t}")
for label, rows in sorted(by_label.items()):
    print(f"[{label}] {len(rows)}개")
    for row in rng.sample(rows, min(6, len(rows))):
        print("   ", row)

[ball] 2773개
    2구, 볼이고요.  →  2구, 바깥쪽으로 살짝 벗어납니다.
    몸쪽 공 골라냅니다.  →  몸쪽 공을 골라내고요.
    147킬로 직구, 낮게 빠집니다.  →  147킬로 직구, 낮고 빠르게 들어옵니다.
    자, 볼입니다.  →  바깥쪽 몸쪽, 볼입니다.
    자, 체인지업, 높은 볼이고요.  →  체인지업, 높게 떠서 볼이에요.
    147킬로 직구, 볼입니다.  →  직구가 몸 바깥으로, 볼.
[double_play] 33개
    자, 더블 플레이!  →  바로 이어지는 땅볼, 더블 플레이 완성입니다.
    더블 플레이!  →  이 타석, 더블 플레이로 연결됐습니다!
    자, 땅볼, 병살로 연결됩니다.  →  자, 땅볼이 나왔고 병살 타구로 연결됐습니다.
    직구, 땅볼, 병살로 연결됩니다.  →  직구를 받고 땅볼, 병살타로 연결됩니다.
    자, 147킬로 직구, 땅볼, 병살로 연결됩니다.  →  직구가 떨어지면서 땅볼이 나오고, 병살 아웃입니다.
    자, 7구, 더블 플레이!  →  7구째, 더블 플레이! 투수의 집중력이 대단합니다!
[foul] 1137개
    131킬로 슬라이더, 빗맞으면서 파울.  →  131킬로 슬라이더, 바깥쪽으로 빠져 나가면서 파울.
    145킬로 직구, 먹혔고 파울입니다.  →  145킬로 직구, 몸쪽으로 박히지만 파울.
    파울이고요.  →  오른쪽으로 빠져나가는 파울.
    147킬로 투심, 파울이 됐고요.  →  147킬로 투심, 뒤로 걷어냅니다. 파울.
    투심, 파울이 됐고요.  →  투심, 왼쪽 파울입니다.
    150킬로 직구, 먹혔고 파울입니다.  →  150킬로 직구, 타자 쳤지만 뒤쪽으로 나가면서 파울.
[none] 4221개
    스트라이크 존을 넓게 쓰고 있어요.  →  스트라이크 존을 확실히 쓰고 있네요.
    아까 그 공은 볼 판정이 조금 아쉬웠어요.  →  그 공은 볼로 판정했지만, 애매했어요.
    끝까지 공을 잘

In [4]:
# 사람 점검용 표본: 반영된 의역 100줄(라벨별 할당) → 블라인드 판정 웹페이지
# GPU 필요 없음. 셀 2(공통 준비)만 먼저 실행하면 된다
# 결과는 동기화 폴더 notebooks/audit/ → 로컬에서 audit.html을 브라우저로 열어 판정
from kbolisten import audit

with (OUT / "scripts.jsonl").open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]
items = audit.sample_items(rows, {c["clip_id"]: c for c in picked})

AUDIT = Path(glob.glob("/content/drive/Othercomputers/*/notebooks")[0]) / "audit"
AUDIT.mkdir(exist_ok=True)
audit.write_page(items, AUDIT / "audit.html")  # 정답 없이 문장만
(AUDIT / "audit_key.json").write_text(json.dumps(items, ensure_ascii=False, indent=1), encoding="utf-8")  # 채점용 정답
print(len(items), "줄 →", AUDIT)
print(Counter(it["label"] for it in items))

100 줄 → /content/drive/Othercomputers/내 컴퓨터/notebooks/audit
Counter({'none': 25, 'ball': 15, 'foul': 12, 'strike': 12, 'out': 10, 'strikeout': 8, 'reach': 8, 'walk': 6, 'double_play': 4})
